In [1]:
# ======================================================================
# T1.3 — CELL 1: ENDPOINT TAXONOMY + DATA INTEGRITY
# ======================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.3 — CELL 1: ENDPOINT TAXONOMY + DATA INTEGRITY")
print("=" * 70)

# ----------------------------------------------------------------------
# 0. Load locked final-checkpoint endpoint table
# ----------------------------------------------------------------------

CSV_PATH = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/results/"
    "model_endpoints_final_checkpoint.csv"
)

df = pd.read_csv(CSV_PATH)

assert len(df) == 30

# ----------------------------------------------------------------------
# 1. Locked design checks
# ----------------------------------------------------------------------

RHO_LEVELS = np.array(
    [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
)

assert np.allclose(
    np.sort(df["rho"].unique()),
    RHO_LEVELS
)

counts = (
    df.groupby("rho")
      .size()
      .to_numpy()
)

assert np.all(counts == 5)

assert df["model"].nunique() == 30

print("\n✓ 30 models loaded")
print("✓ Six rho levels present")
print("✓ Five replicates per rho")
print("✓ All model identifiers unique")

# ----------------------------------------------------------------------
# 2. Endpoint taxonomy
#
# IMPORTANT:
#
# We distinguish:
#
# A. PRIMARY MECHANISTIC SIGNAL
#    delta_z_hair
#
# B. SCALE / DECISION TRANSFORMATIONS
#    delta_z_norm
#    delta_p
#    flip_fraction
#
# C. DOWNSTREAM BEHAVIORAL CONSEQUENCES
#    D
#    D_AUROC
#
# D. ALGEBRAICALLY COUPLED BEHAVIORAL QUANTITY
#    BA_neutral_minus_reversed
#
# E. AUXILIARY SPATIAL MECHANISTIC EVIDENCE
#    xai_enrichment
# ----------------------------------------------------------------------

taxonomy = pd.DataFrame([
    {
        "endpoint": "delta_z_hair",
        "class": "primary_mechanistic_signal",
        "role":
            "Continuous clean→hair logit perturbation",
        "independence_note":
            "Primary shortcut-reliance measure"
    },

    {
        "endpoint": "delta_z_norm",
        "class": "scale_transformation",
        "role":
            "Δz normalized by clean-logit SD",
        "independence_note":
            "Same underlying logit response; not independent"
    },

    {
        "endpoint": "delta_p",
        "class": "scale_transformation",
        "role":
            "Clean→hair probability change",
        "independence_note":
            "Derived from the same logits"
    },

    {
        "endpoint": "flip_fraction",
        "class": "decision_transformation",
        "role":
            "Fraction of clean predictions changed by hair insertion",
        "independence_note":
            "Thresholded manifestation of the logit perturbation"
    },

    {
        "endpoint": "D",
        "class": "downstream_behavior",
        "role":
            "BA_clean − BA_reversed",
        "independence_note":
            "Primary downstream behavioral consequence"
    },

    {
        "endpoint": "D_AUROC",
        "class": "downstream_behavior",
        "role":
            "AUROC_clean − AUROC_reversed",
        "independence_note":
            "Complementary ranking-based consequence"
    },

    {
        "endpoint": "BA_neutral_minus_reversed",
        "class": "algebraically_coupled",
        "role":
            "BA_neutral − BA_reversed",
        "independence_note":
            "Shares BA_reversed with D; not independent evidence"
    },

    {
        "endpoint": "xai_enrichment",
        "class": "auxiliary_mechanistic_evidence",
        "role":
            "Integrated-Gradients attribution enrichment in hair region",
        "independence_note":
            "Separate spatial attribution measurement; requires XAI validation"
    },
])

print("\n" + "=" * 70)
print("PART A — ENDPOINT TAXONOMY")
print("=" * 70)

print(
    taxonomy.to_string(
        index=False
    )
)

# ----------------------------------------------------------------------
# 3. Verify mathematical relationships already established
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("PART B — MATHEMATICAL RELATIONSHIP CHECKS")
print("=" * 70)

# D
D_reconstructed = (
    df["BA_clean"]
    -
    df["BA_reversed"]
)

err_D = np.max(
    np.abs(
        df["D"].to_numpy()
        -
        D_reconstructed.to_numpy()
    )
)

print(
    f"D reconstruction max error: "
    f"{err_D:.3e}"
)

# BA neutral minus reversed
BA_diff_reconstructed = (
    df["BA_neutral"]
    -
    df["BA_reversed"]
)

err_BA_diff = np.max(
    np.abs(
        df["BA_neutral_minus_reversed"].to_numpy()
        -
        BA_diff_reconstructed.to_numpy()
    )
)

print(
    "BA_neutral_minus_reversed "
    f"reconstruction max error: "
    f"{err_BA_diff:.3e}"
)

# D decomposition
D_decomp = (
    df["BA_neutral_minus_reversed"]
    +
    (
        df["BA_clean"]
        -
        df["BA_neutral"]
    )
)

err_decomp = np.max(
    np.abs(
        df["D"].to_numpy()
        -
        D_decomp.to_numpy()
    )
)

print(
    "D decomposition max error: "
    f"{err_decomp:.3e}"
)

assert err_D < 1e-12
assert err_BA_diff < 1e-12
assert err_decomp < 1e-12

print("\n✓ Algebraic identities reproduced")

# ----------------------------------------------------------------------
# 4. Check finite values
# ----------------------------------------------------------------------

ANALYSIS_ENDPOINTS = [
    "delta_z_hair",
    "delta_z_norm",
    "delta_p",
    "flip_fraction",
    "D",
    "D_AUROC",
    "BA_neutral_minus_reversed",
    "xai_enrichment",
]

finite_table = pd.DataFrame({
    endpoint: [
        np.isfinite(
            df[endpoint].to_numpy(
                dtype=float
            )
        ).all()
    ]
    for endpoint in ANALYSIS_ENDPOINTS
}).T

finite_table.columns = [
    "all_finite"
]

print("\n" + "=" * 70)
print("PART C — FINITENESS CHECK")
print("=" * 70)

print(
    finite_table.to_string()
)

assert finite_table["all_finite"].all()

# ----------------------------------------------------------------------
# 5. Save taxonomy / integrity record
# ----------------------------------------------------------------------

OUT = Path(
    "/kaggle/working/t1_3_results"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

taxonomy.to_csv(
    OUT / "endpoint_taxonomy.csv",
    index=False
)

finite_table.to_csv(
    OUT / "endpoint_finiteness.csv"
)

integrity = {
    "n_models": int(len(df)),
    "n_rho_levels": int(df["rho"].nunique()),
    "replicates_per_rho": 5,
    "all_model_ids_unique":
        bool(df["model"].nunique() == 30),
    "D_identity_error":
        float(err_D),
    "BA_difference_identity_error":
        float(err_BA_diff),
    "D_decomposition_error":
        float(err_decomp),
}

import json

with open(
    OUT / "T1_3_integrity.json",
    "w"
) as f:
    json.dump(
        integrity,
        f,
        indent=2
    )

print("\n" + "=" * 70)
print("CELL 1 COMPLETE")
print("=" * 70)

print("""
T1.3 endpoint taxonomy is locked.

Next:

CELL 2 will examine within-rho cross-model relationships.

This is critical because pooled correlations can be dominated
by the common rho dose-response. The mechanistic question is
whether models that differ in Δz_hair WITHIN THE SAME rho level
also differ systematically in downstream behavioral harm.
""")

T1.3 — CELL 1: ENDPOINT TAXONOMY + DATA INTEGRITY

✓ 30 models loaded
✓ Six rho levels present
✓ Five replicates per rho
✓ All model identifiers unique

PART A — ENDPOINT TAXONOMY
                 endpoint                          class                                                       role                                                 independence_note
             delta_z_hair     primary_mechanistic_signal                   Continuous clean→hair logit perturbation                                 Primary shortcut-reliance measure
             delta_z_norm           scale_transformation                            Δz normalized by clean-logit SD                   Same underlying logit response; not independent
                  delta_p           scale_transformation                              Clean→hair probability change                                      Derived from the same logits
            flip_fraction        decision_transformation    Fraction of clean predictions ch

In [2]:
# ================================================================
# T1.3 — CELL 2: WITHIN-RHO CROSS-MODEL RELATIONSHIPS
# ================================================================
#
# Question:
# At a fixed rho, do replicate-to-replicate differences in
# delta_z_hair track differences in downstream behavioral harm?
#
# IMPORTANT:
# n = 5 models per rho.
# These correlations are diagnostic/mechanistic, NOT confirmatory
# significance tests.
# ================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

# ------------------------------------------------
# Load locked endpoint table
# ------------------------------------------------
CSV_PATH = "/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/results/model_endpoints_final_checkpoint.csv"

df = pd.read_csv(CSV_PATH)

assert len(df) == 30
assert df["model"].nunique() == 30

ENDPOINTS = [
    "delta_z_hair",
    "delta_z_norm",
    "delta_p",
    "flip_fraction",
    "D",
    "D_AUROC",
    "BA_neutral_minus_reversed",
    "xai_enrichment",
]

PRIMARY = "delta_z_hair"

TARGETS = [
    "delta_z_norm",
    "delta_p",
    "flip_fraction",
    "D",
    "D_AUROC",
    "BA_neutral_minus_reversed",
    "xai_enrichment",
]

RHO_LEVELS = sorted(df["rho"].unique())

# ------------------------------------------------
# Within-rho correlation function
# ------------------------------------------------
def corr_at_each_rho(x_col, y_col):

    rows = []

    for rho in RHO_LEVELS:

        sub = (
            df[df["rho"] == rho]
            .sort_values("replicate")
            .reset_index(drop=True)
        )

        assert len(sub) == 5, (
            f"Expected 5 replicates at rho={rho}, "
            f"found {len(sub)}"
        )

        x = sub[x_col].to_numpy(dtype=float)
        y = sub[y_col].to_numpy(dtype=float)

        pearson_r, pearson_p = pearsonr(x, y)
        spearman_r, spearman_p = spearmanr(x, y)

        rows.append({
            "rho": rho,
            "n": len(sub),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_r,
            "spearman_p": spearman_p,
        })

    return pd.DataFrame(rows)


# ------------------------------------------------
# Run all within-rho relationships
# ------------------------------------------------
results = []

for target in TARGETS:

    tmp = corr_at_each_rho(PRIMARY, target)

    for _, row in tmp.iterrows():
        results.append({
            "x": PRIMARY,
            "y": target,
            **row.to_dict()
        })

within_df = pd.DataFrame(results)


# ------------------------------------------------
# Print each relationship
# ------------------------------------------------
print("=" * 70)
print("T1.3 — WITHIN-RHO CROSS-MODEL RELATIONSHIPS")
print("=" * 70)

for target in TARGETS:

    sub = within_df[within_df["y"] == target]

    print(f"\nΔz_hair  ↔  {target}")
    print("-" * 70)

    print(
        sub[
            [
                "rho",
                "pearson_r",
                "spearman_rho",
            ]
        ].to_string(index=False)
    )

    s = sub["spearman_rho"]

    print(
        f"Mean within-rho Spearman:   {s.mean(): .4f}"
    )
    print(
        f"Median within-rho Spearman: {s.median(): .4f}"
    )
    print(
        f"Range:                       "
        f"[{s.min(): .4f}, {s.max(): .4f}]"
    )
    print(
        f"Positive rho levels:         "
        f"{int((s > 0).sum())}/6"
    )


# ------------------------------------------------
# Compact summary table
# ------------------------------------------------
summary_rows = []

for target in TARGETS:

    sub = within_df[within_df["y"] == target]

    s = sub["spearman_rho"]

    summary_rows.append({
        "endpoint": target,
        "mean_spearman": s.mean(),
        "median_spearman": s.median(),
        "min_spearman": s.min(),
        "max_spearman": s.max(),
        "positive_rho_levels": int((s > 0).sum()),
    })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("COMPACT WITHIN-RHO SUMMARY")
print("=" * 70)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------
# Save result
# ------------------------------------------------
OUT_DIR = "/kaggle/working/t1_3_results"
import os
os.makedirs(OUT_DIR, exist_ok=True)

within_df.to_csv(
    f"{OUT_DIR}/within_rho_cross_model_correlations.csv",
    index=False
)

summary_df.to_csv(
    f"{OUT_DIR}/within_rho_cross_model_summary.csv",
    index=False
)


# ------------------------------------------------
# Interpretation guardrail
# ------------------------------------------------
print("\n" + "=" * 70)
print("INTERPRETATION GUARDRAIL")
print("=" * 70)

print(
    "These are within-rho replicate correlations with n=5 per rho."
)
print(
    "They are diagnostic only and should NOT be treated as "
    "independent confirmatory significance tests."
)
print(
    "The main question is whether the direction is reasonably "
    "consistent across rho levels, not whether individual p-values "
    "are significant."
)

print("\n✓ CELL 2 COMPLETE")

T1.3 — WITHIN-RHO CROSS-MODEL RELATIONSHIPS

Δz_hair  ↔  delta_z_norm
----------------------------------------------------------------------
 rho  pearson_r  spearman_rho
 0.0   0.956151           0.9
 0.2   0.791929           0.6
 0.4   0.166902           0.3
 0.6   0.070508          -0.1
 0.8   0.589343           0.6
 1.0   0.326710           0.3
Mean within-rho Spearman:    0.4333
Median within-rho Spearman:  0.4500
Range:                       [-0.1000,  0.9000]
Positive rho levels:         5/6

Δz_hair  ↔  delta_p
----------------------------------------------------------------------
 rho  pearson_r  spearman_rho
 0.0   0.942276           0.9
 0.2   0.000739          -0.1
 0.4  -0.372625           0.1
 0.6   0.349101           0.3
 0.8   0.742352           0.9
 1.0   0.190500           0.3
Mean within-rho Spearman:    0.4000
Median within-rho Spearman:  0.3000
Range:                       [-0.1000,  0.9000]
Positive rho levels:         5/6

Δz_hair  ↔  flip_fraction
--------------

In [3]:
# ================================================================
# T1.3 — CELL 3: PARTIAL SPEARMAN RELATIONSHIPS CONTROLLING FOR ρ
# ================================================================
#
# Question:
# After removing the common monotonic rho dose-response, do
# residual differences in delta_z_hair remain associated with
# residual differences in downstream / auxiliary endpoints?
#
# Method:
# 1. Rank-transform x, y, and rho.
# 2. Regress ranked x on ranked rho.
# 3. Regress ranked y on ranked rho.
# 4. Correlate the two residual vectors.
#
# This is the standard rank-based partial-Spearman construction.
#
# IMPORTANT:
# This is an exploratory mechanistic audit, NOT a confirmatory
# hypothesis test. No endpoint-wise multiplicity correction is
# used for interpretation here.
# ================================================================

import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

CSV_PATH = (
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/results/"
    "model_endpoints_final_checkpoint.csv"
)

df = pd.read_csv(CSV_PATH)

assert len(df) == 30
assert df["model"].nunique() == 30

PRIMARY = "delta_z_hair"

TARGETS = [
    "delta_z_norm",
    "delta_p",
    "flip_fraction",
    "D",
    "D_AUROC",
    "BA_neutral_minus_reversed",
    "xai_enrichment",
]

# ------------------------------------------------
# Rank-residual function
# ------------------------------------------------
def rank_residual(values, rho):
    """
    Rank-transform values and rho, then remove the linear
    relationship with ranked rho.

    Returns residuals in rank space.
    """
    y_rank = pd.Series(values).rank(method="average").to_numpy(dtype=float)
    r_rank = pd.Series(rho).rank(method="average").to_numpy(dtype=float)

    # Design matrix with intercept
    X = np.column_stack([
        np.ones(len(r_rank)),
        r_rank
    ])

    beta = np.linalg.lstsq(X, y_rank, rcond=None)[0]
    residual = y_rank - X @ beta

    return residual


# ------------------------------------------------
# Partial Spearman
# ------------------------------------------------
def partial_spearman(x, y, rho):
    rx = rank_residual(x, rho)
    ry = rank_residual(y, rho)

    r = np.corrcoef(rx, ry)[0, 1]

    return float(r)


# ------------------------------------------------
# Permutation test
#
# Null:
# after controlling for rho, x and y are unrelated.
#
# We permute the residual rank vector of y while keeping
# the residual rank vector of x fixed.
#
# This is diagnostic only.
# ------------------------------------------------
N_PERM = 100_000
PERM_SEED = 314159

rng = np.random.default_rng(PERM_SEED)


def partial_spearman_permutation(x, y, rho, n_perm=N_PERM):

    rx = rank_residual(x, rho)
    ry = rank_residual(y, rho)

    observed = np.corrcoef(rx, ry)[0, 1]

    null = np.empty(n_perm, dtype=float)

    for i in range(n_perm):
        perm = rng.permutation(ry)
        null[i] = np.corrcoef(rx, perm)[0, 1]

    p = (
        np.sum(np.abs(null) >= abs(observed)) + 1
    ) / (n_perm + 1)

    return float(observed), float(p)


# ------------------------------------------------
# Run relationships
# ------------------------------------------------
rows = []

rho = df["rho"].to_numpy(dtype=float)

for target in TARGETS:

    x = df[PRIMARY].to_numpy(dtype=float)
    y = df[target].to_numpy(dtype=float)

    r_partial, p_perm = partial_spearman_permutation(
        x,
        y,
        rho,
        n_perm=N_PERM
    )

    rows.append({
        "x": PRIMARY,
        "y": target,
        "partial_spearman": r_partial,
        "permutation_p": p_perm
    })


results = pd.DataFrame(rows)

# ------------------------------------------------
# Sort by absolute residual association
# ------------------------------------------------
results["abs_partial_spearman"] = (
    results["partial_spearman"].abs()
)

results = (
    results
    .sort_values(
        "abs_partial_spearman",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------
# Print
# ------------------------------------------------
print("=" * 70)
print("T1.3 — PARTIAL SPEARMAN RELATIONSHIPS CONTROLLING FOR ρ")
print("=" * 70)

print(
    results[
        [
            "y",
            "partial_spearman",
            "permutation_p"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

print(
    "These coefficients measure residual association after "
    "removing the common monotonic rho trend."
)

print(
    "They are exploratory mechanistic diagnostics, not "
    "confirmatory endpoint tests."
)

print(
    "Do NOT interpret a large pooled correlation as independent "
    "evidence if the partial association collapses after "
    "controlling for rho."
)

# ------------------------------------------------
# Save
# ------------------------------------------------
OUT_DIR = "/kaggle/working/t1_3_results"
os.makedirs(OUT_DIR, exist_ok=True)

results.to_csv(
    f"{OUT_DIR}/partial_spearman_rho_controlled.csv",
    index=False
)

print("\n✓ CELL 3 COMPLETE")

T1.3 — PARTIAL SPEARMAN RELATIONSHIPS CONTROLLING FOR ρ
                        y  partial_spearman  permutation_p
           xai_enrichment          0.478379       0.008000
             delta_z_norm          0.433333       0.017910
                  delta_p          0.400000       0.030690
                  D_AUROC          0.383333       0.037910
                        D          0.216667       0.246798
            flip_fraction          0.183333       0.329547
BA_neutral_minus_reversed         -0.050000       0.792522

INTERPRETATION
These coefficients measure residual association after removing the common monotonic rho trend.
They are exploratory mechanistic diagnostics, not confirmatory endpoint tests.
Do NOT interpret a large pooled correlation as independent evidence if the partial association collapses after controlling for rho.

✓ CELL 3 COMPLETE


In [4]:
# ================================================================
# T1.3 — CELL 4: CROSS-ENDPOINT REGRESSION AUDIT
# ================================================================
#
# Question:
# Does delta_z_hair explain residual variation in downstream
# behavioral harm after accounting for rho?
#
# Models:
#   M0: outcome ~ rho
#   M1: outcome ~ rho + delta_z_hair
#
# Report:
#   - R² of rho-only model
#   - R² of rho + delta_z model
#   - incremental ΔR²
#   - delta_z coefficient
#   - standard error
#   - nominal p-value
#
# This is associational only.
# It does NOT establish causality.
# ================================================================

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

CSV_PATH = (
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/results/model_endpoints_final_checkpoint.csv"
)

df = pd.read_csv(CSV_PATH)

# ------------------------------------------------
# DATA INTEGRITY
# ------------------------------------------------

assert len(df) == 30, f"Expected 30 models, found {len(df)}"
assert df["model"].nunique() == 30, "Model identifiers are not unique"

expected_rho = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
assert sorted(df["rho"].unique().tolist()) == expected_rho

for rho in expected_rho:
    assert (df["rho"] == rho).sum() == 5, (
        f"Expected 5 replicates at rho={rho}"
    )

PRIMARY = "delta_z_hair"

OUTCOMES = [
    "D",
    "D_AUROC",
    "xai_enrichment",
]

# Check required columns
required = ["rho", PRIMARY] + OUTCOMES

missing = [
    c for c in required
    if c not in df.columns
]

assert not missing, f"Missing required columns: {missing}"

# Check finite values
for c in required:
    assert np.isfinite(df[c].to_numpy(dtype=float)).all(), (
        f"Non-finite values found in {c}"
    )


# ------------------------------------------------
# REGRESSION ANALYSIS
# ------------------------------------------------

results = []

print("=" * 70)
print("T1.3 — CROSS-ENDPOINT REGRESSION AUDIT")
print("=" * 70)

print("\nData:")
print(f"Models:       {len(df)}")
print(f"Rho levels:   {len(expected_rho)}")
print("Replicates:   5 per rho")
print("Primary X:    delta_z_hair")

for outcome in OUTCOMES:

    y = df[outcome].to_numpy(dtype=float)

    # ------------------------------------------------------------
    # MODEL 0
    # outcome ~ rho
    # ------------------------------------------------------------

    X0 = sm.add_constant(
        df[["rho"]].to_numpy(dtype=float)
    )

    m0 = sm.OLS(y, X0).fit()

    # ------------------------------------------------------------
    # MODEL 1
    # outcome ~ rho + delta_z_hair
    # ------------------------------------------------------------

    X1 = sm.add_constant(
        df[["rho", PRIMARY]].to_numpy(dtype=float)
    )

    m1 = sm.OLS(y, X1).fit()

    # ------------------------------------------------------------
    # Extract statistics
    # ------------------------------------------------------------

    r2_0 = float(m0.rsquared)
    r2_1 = float(m1.rsquared)

    delta_r2 = r2_1 - r2_0

    # Parameter order:
    # const, rho, delta_z_hair
    beta_rho = float(m1.params[1])
    beta_dz = float(m1.params[2])

    se_rho = float(m1.bse[1])
    se_dz = float(m1.bse[2])

    p_rho = float(m1.pvalues[1])
    p_dz = float(m1.pvalues[2])

    ci_dz_low, ci_dz_high = m1.conf_int()[2]

    results.append({
        "outcome": outcome,
        "R2_rho_only": r2_0,
        "R2_rho_plus_delta_z": r2_1,
        "delta_R2": delta_r2,
        "beta_rho": beta_rho,
        "beta_delta_z": beta_dz,
        "SE_delta_z": se_dz,
        "p_delta_z": p_dz,
        "CI95_delta_z_low": float(ci_dz_low),
        "CI95_delta_z_high": float(ci_dz_high),
    })

    # ------------------------------------------------------------
    # PRINT RESULT
    # ------------------------------------------------------------

    print("\n" + "-" * 70)
    print(f"OUTCOME: {outcome}")
    print("-" * 70)

    print(
        f"R² (rho only):          {r2_0:.6f}"
    )

    print(
        f"R² (rho + Δz):          {r2_1:.6f}"
    )

    print(
        f"Incremental ΔR²:        {delta_r2:.6f}"
    )

    print(
        f"rho coefficient:        {beta_rho:.6f}"
    )

    print(
        f"Δz coefficient:         {beta_dz:.6f}"
    )

    print(
        f"Δz standard error:      {se_dz:.6f}"
    )

    print(
        f"Δz nominal p-value:     {p_dz:.6f}"
    )

    print(
        f"Δz 95% CI:              "
        f"[{float(ci_dz_low):.6f}, {float(ci_dz_high):.6f}]"
    )

    # ------------------------------------------------------------
    # Explicit coefficient table
    # ------------------------------------------------------------

    coef_table = pd.DataFrame({
        "term": [
            "Intercept",
            "rho",
            "delta_z_hair",
        ],
        "coefficient": [
            float(m1.params[0]),
            float(m1.params[1]),
            float(m1.params[2]),
        ],
        "SE": [
            float(m1.bse[0]),
            float(m1.bse[1]),
            float(m1.bse[2]),
        ],
        "p_value": [
            float(m1.pvalues[0]),
            float(m1.pvalues[1]),
            float(m1.pvalues[2]),
        ],
    })

    print("\nFull model coefficients:")
    print(
        coef_table.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ------------------------------------------------
# SUMMARY TABLE
# ------------------------------------------------

reg_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("REGRESSION SUMMARY")
print("=" * 70)

print(
    reg_df[
        [
            "outcome",
            "R2_rho_only",
            "R2_rho_plus_delta_z",
            "delta_R2",
            "beta_delta_z",
            "SE_delta_z",
            "p_delta_z",
            "CI95_delta_z_low",
            "CI95_delta_z_high",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------

OUT_DIR = "/kaggle/working/t1_3_results"
os.makedirs(OUT_DIR, exist_ok=True)

reg_df.to_csv(
    f"{OUT_DIR}/cross_endpoint_regression.csv",
    index=False
)

print("\n✓ Results saved:")
print(
    f"  {OUT_DIR}/cross_endpoint_regression.csv"
)

print("\n✓ CELL 4 COMPLETE")

T1.3 — CROSS-ENDPOINT REGRESSION AUDIT

Data:
Models:       30
Rho levels:   6
Replicates:   5 per rho
Primary X:    delta_z_hair

----------------------------------------------------------------------
OUTCOME: D
----------------------------------------------------------------------
R² (rho only):          0.963369
R² (rho + Δz):          0.977039
Incremental ΔR²:        0.013671
rho coefficient:        0.033380
Δz coefficient:         0.014294
Δz standard error:      0.003565
Δz nominal p-value:     0.000432
Δz 95% CI:              [0.006979, 0.021609]

Full model coefficients:
        term  coefficient       SE  p_value
   Intercept     0.004414 0.008719 0.616790
         rho     0.033380 0.035494 0.355328
delta_z_hair     0.014294 0.003565 0.000432

----------------------------------------------------------------------
OUTCOME: D_AUROC
----------------------------------------------------------------------
R² (rho only):          0.978095
R² (rho + Δz):          0.983696
Incremental 

In [5]:
# ================================================================
# T1.3 — FINAL SYNTHESIS / DECISION RECORD
# ================================================================
#
# Purpose:
# Consolidate the mechanistic audit without introducing new
# statistical tests.
#
# T1.3 asks:
#   1. Are pooled endpoint relationships mostly explained by rho?
#   2. Do residual relationships remain after controlling for rho?
#   3. Which endpoints provide genuinely distinct evidence?
#
# No new hypothesis testing is performed here.
# ================================================================

import os
import json
import numpy as np
import pandas as pd

OUT_DIR = "/kaggle/working/t1_3_results"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------
# Load previously generated results
# ------------------------------------------------

within_path = (
    f"{OUT_DIR}/within_rho_cross_model_summary.csv"
)

partial_path = (
    f"{OUT_DIR}/partial_spearman_rho_controlled.csv"
)

reg_path = (
    f"{OUT_DIR}/cross_endpoint_regression.csv"
)

within_df = pd.read_csv(within_path)
partial_df = pd.read_csv(partial_path)
reg_df = pd.read_csv(reg_path)

# ------------------------------------------------
# Endpoint evidence taxonomy
# ------------------------------------------------

evidence_taxonomy = pd.DataFrame([
    {
        "endpoint": "delta_z_hair",
        "evidence_role": "PRIMARY",
        "classification": "Primary mechanistic signal",
        "interpretation":
            "Continuous clean-to-hair logit perturbation; "
            "primary measure of model response to the synthetic artifact."
    },
    {
        "endpoint": "delta_z_norm",
        "evidence_role": "SUPPORTING",
        "classification": "Scale transformation",
        "interpretation":
            "Normalized version of the same logit perturbation; "
            "not independent evidence."
    },
    {
        "endpoint": "delta_p",
        "evidence_role": "SUPPORTING",
        "classification": "Scale transformation",
        "interpretation":
            "Probability-scale manifestation of the same clean-to-hair "
            "logit response; not independent evidence."
    },
    {
        "endpoint": "flip_fraction",
        "evidence_role": "SUPPORTING",
        "classification": "Decision transformation",
        "interpretation":
            "Thresholded behavioral manifestation of clean-to-hair "
            "prediction changes; not independent mechanistic evidence."
    },
    {
        "endpoint": "D",
        "evidence_role": "PRIMARY_DOWNSTREAM",
        "classification": "Primary downstream consequence",
        "interpretation":
            "Balanced-accuracy degradation under reversed association."
    },
    {
        "endpoint": "D_AUROC",
        "evidence_role": "SECONDARY_DOWNSTREAM",
        "classification": "Complementary downstream consequence",
        "interpretation":
            "Ranking-based performance degradation under reversed association."
    },
    {
        "endpoint": "BA_neutral_minus_reversed",
        "evidence_role": "EXCLUDE_AS_INDEPENDENT",
        "classification": "Algebraically coupled",
        "interpretation":
            "Shares BA_reversed with D and is not independent evidence."
    },
    {
        "endpoint": "xai_enrichment",
        "evidence_role": "AUXILIARY_MECHANISTIC",
        "classification": "Spatial attribution evidence",
        "interpretation":
            "Integrated-Gradients enrichment in the synthetic-hair region; "
            "requires the planned XAI sanity checks."
    },
])

print("=" * 70)
print("T1.3 — FINAL MECHANISTIC AUDIT SYNTHESIS")
print("=" * 70)

print("\nENDPOINT EVIDENCE TAXONOMY")
print("-" * 70)

print(
    evidence_taxonomy.to_string(index=False)
)

# ------------------------------------------------
# Extract key numerical findings
# ------------------------------------------------

def get_partial(endpoint):
    row = partial_df[
        partial_df["y"] == endpoint
    ].iloc[0]

    return (
        float(row["partial_spearman"]),
        float(row["permutation_p"])
    )


def get_reg(endpoint):
    row = reg_df[
        reg_df["outcome"] == endpoint
    ].iloc[0]

    return {
        "R2_rho_only": float(row["R2_rho_only"]),
        "R2_rho_plus_delta_z": float(
            row["R2_rho_plus_delta_z"]
        ),
        "delta_R2": float(row["delta_R2"]),
        "beta_delta_z": float(row["beta_delta_z"]),
        "p_delta_z": float(row["p_delta_z"]),
    }


partial_D = get_partial("D")
partial_D_AUROC = get_partial("D_AUROC")
partial_IG = get_partial("xai_enrichment")

reg_D = get_reg("D")
reg_D_AUROC = get_reg("D_AUROC")
reg_IG = get_reg("xai_enrichment")

# ------------------------------------------------
# Print mechanistic findings
# ------------------------------------------------

print("\n" + "=" * 70)
print("KEY MECHANISTIC FINDINGS")
print("=" * 70)

print(
    "\n1. COMMON rho DOSE-RESPONSE"
)

print(
    "The very strong pooled endpoint correlations are largely "
    "associated with the shared rho gradient."
)

print(
    "\n2. WITHIN-rho REPLICATION"
)

for endpoint in ["D", "D_AUROC", "xai_enrichment"]:

    row = within_df[
        within_df["endpoint"] == endpoint
    ].iloc[0]

    print(
        f"{endpoint}: "
        f"mean within-rho Spearman="
        f"{float(row['mean_spearman']):.4f}, "
        f"range="
        f"[{float(row['min_spearman']):.4f}, "
        f"{float(row['max_spearman']):.4f}], "
        f"positive levels="
        f"{int(row['positive_rho_levels'])}/6"
    )

print(
    "\n3. rho-CONTROLLED ASSOCIATIONS"
)

print(
    f"Δz ↔ D: "
    f"partial Spearman={partial_D[0]:.4f}, "
    f"permutation p={partial_D[1]:.6f}"
)

print(
    f"Δz ↔ D_AUROC: "
    f"partial Spearman={partial_D_AUROC[0]:.4f}, "
    f"permutation p={partial_D_AUROC[1]:.6f}"
)

print(
    f"Δz ↔ IG: "
    f"partial Spearman={partial_IG[0]:.4f}, "
    f"permutation p={partial_IG[1]:.6f}"
)

print(
    "\n4. INCREMENTAL REGRESSION CONTRIBUTION"
)

for name, reg in [
    ("D", reg_D),
    ("D_AUROC", reg_D_AUROC),
    ("xai_enrichment", reg_IG),
]:

    print(
        f"{name}: "
        f"ΔR²={reg['delta_R2']:.6f}, "
        f"β_Δz={reg['beta_delta_z']:.6f}, "
        f"nominal p={reg['p_delta_z']:.6f}"
    )

# ------------------------------------------------
# Final evidence classification
# ------------------------------------------------

decision = {
    "T1_3_status": "COMPLETE",

    "pooled_correlations_interpretation":
        "Pooled endpoint correlations are strongly influenced "
        "by the common rho dose-response and must not be treated "
        "as independent evidence.",

    "primary_mechanistic_signal":
        "delta_z_hair",

    "primary_downstream_consequence":
        "D",

    "complementary_downstream_consequence":
        "D_AUROC",

    "auxiliary_mechanistic_evidence":
        "xai_enrichment",

    "non_independent_scale_transformations": [
        "delta_z_norm",
        "delta_p",
    ],

    "non_independent_decision_transformation": [
        "flip_fraction",
    ],

    "algebraically_coupled_endpoint": [
        "BA_neutral_minus_reversed",
    ],

    "rho_controlled_partial_spearman": {
        "D": partial_D[0],
        "D_AUROC": partial_D_AUROC[0],
        "xai_enrichment": partial_IG[0],
    },

    "incremental_R2": {
        "D": reg_D["delta_R2"],
        "D_AUROC": reg_D_AUROC["delta_R2"],
        "xai_enrichment": reg_IG["delta_R2"],
    },

    "mechanistic_claim_level":
        "Residual associations remain after accounting for rho, "
        "but they are substantially weaker than pooled associations. "
        "The results support association between model-level hair "
        "sensitivity and downstream/attribution endpoints, but do "
        "not establish a deterministic or causal relationship.",

    "xai_claim_status":
        "Auxiliary only until planned XAI sanity checks are completed.",

    "next_tier":
        "T1.4 XAI sanity checks",
}

# ------------------------------------------------
# Save machine-readable decision record
# ------------------------------------------------

decision_path = (
    f"{OUT_DIR}/T1_3_DECISION_RECORD.json"
)

with open(decision_path, "w") as f:
    json.dump(
        decision,
        f,
        indent=2
    )

# Save taxonomy
evidence_taxonomy.to_csv(
    f"{OUT_DIR}/T1_3_ENDPOINT_EVIDENCE_TAXONOMY.csv",
    index=False
)

# ------------------------------------------------
# Final print
# ------------------------------------------------

print("\n" + "=" * 70)
print("T1.3 DECISION")
print("=" * 70)

print(
    "✓ T1.3 COMPLETE"
)

print(
    "\nPrimary interpretation:"
)

print(
    "The common rho manipulation explains most of the pooled "
    "cross-endpoint covariance. After controlling for rho, "
    "residual associations remain between Δz_hair and D, "
    "D_AUROC, and IG enrichment, but these relationships are "
    "substantially weaker than the pooled correlations."
)

print(
    "\nEvidence hierarchy:"
)

print(
    "  PRIMARY MECHANISTIC:       delta_z_hair"
)

print(
    "  PRIMARY DOWNSTREAM:        D"
)

print(
    "  COMPLEMENTARY DOWNSTREAM:  D_AUROC"
)

print(
    "  AUXILIARY MECHANISTIC:     xai_enrichment"
)

print(
    "  TRANSFORMATIONS:           delta_z_norm, delta_p, flip_fraction"
)

print(
    "  ALGEBRAICALLY COUPLED:     BA_neutral_minus_reversed"
)

print(
    "\n✓ Decision record saved:"
)

print(
    f"  {decision_path}"
)

print(
    "\nNext tier:"
)

print(
    "  T1.4 — XAI sanity checks"
)

print("\n✓ CELL COMPLETE")

T1.3 — FINAL MECHANISTIC AUDIT SYNTHESIS

ENDPOINT EVIDENCE TAXONOMY
----------------------------------------------------------------------
                 endpoint          evidence_role                       classification                                                                                                  interpretation
             delta_z_hair                PRIMARY           Primary mechanistic signal       Continuous clean-to-hair logit perturbation; primary measure of model response to the synthetic artifact.
             delta_z_norm             SUPPORTING                 Scale transformation                                    Normalized version of the same logit perturbation; not independent evidence.
                  delta_p             SUPPORTING                 Scale transformation             Probability-scale manifestation of the same clean-to-hair logit response; not independent evidence.
            flip_fraction             SUPPORTING              Decisi